In [55]:
import pandas as pd
import numpy as np

In [ ]:
#🎯 Goal

#To build an accurate and robust machine learning model that predicts food delivery time using real-world operational, environmental, and delivery partner data.

#📌 Objectives
#Understand and explore the dataset to identify key factors affecting delivery time
#Perform data cleaning and handle missing values effectively
#Engineer meaningful features such as traffic impact, peak hours, and preparation time
#Avoid data leakage and ensure model reliability
#Train and compare multiple models (Linear Regression, Random Forest, XGBoost)
#Optimize model performance using hyperparameter tuning
#Validate model stability using cross-validation
#Derive actionable insights to improve delivery efficiency

In [56]:
# LOADING THE DATA

eta=pd.read_csv(r'C:\Users\HP\Desktop\Bishal_\ML\ML PROJECTS\ZOMATO_DELIVERY_DATA\zomato_cleaned.csv')
eta.head(4)

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken (min),distance_km,delivery_speed
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,12-02-2022,21:55,...,Jam,2,Snack,motorcycle,3.0,No,Metropolitian,46,10.280582,Slow
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,13-02-2022,14:55,...,High,1,Meal,motorcycle,1.0,No,Metropolitian,23,6.242319,Average
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,04-03-2022,17:30,...,Medium,1,Drinks,scooter,1.0,No,Metropolitian,21,13.787860,Average
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,13-02-2022,09:20,...,Low,0,Buffet,motorcycle,0.0,No,Metropolitian,20,2.930258,Average


In [57]:
# BASIC UNDERSTANDING

print("Shape:", eta.shape)

print("\nColumns:\n", eta.columns.tolist())

print("\nInfo:\n")
eta.info()

eta.head(4)

Shape: (38964, 22)

Columns:
 ['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weather_conditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken (min)', 'distance_km', 'delivery_speed']

Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38964 entries, 0 to 38963
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           38964 non-null  object 
 1   Delivery_person_ID           38964 non-null  object 
 2   Delivery_person_Age          37945 non-null  float64
 3   Delivery_person_Ratings      37909 non-null  float64
 4   Restaurant_latitude          38964 non-null  float64
 5   Restau

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken (min),distance_km,delivery_speed
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,12-02-2022,21:55,...,Jam,2,Snack,motorcycle,3.0,No,Metropolitian,46,10.280582,Slow
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,13-02-2022,14:55,...,High,1,Meal,motorcycle,1.0,No,Metropolitian,23,6.242319,Average
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,04-03-2022,17:30,...,Medium,1,Drinks,scooter,1.0,No,Metropolitian,21,13.787860,Average
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,13-02-2022,09:20,...,Low,0,Buffet,motorcycle,0.0,No,Metropolitian,20,2.930258,Average


In [58]:
# DROPPING UNNCESSARY COLUMNS TO STOP THE LEAKAGE IN THE DATA 

df = eta.copy()

df = df.drop(columns=['ID','Delivery_person_ID','delivery_speed'])

df.shape

(38964, 19)

In [59]:
# CONVERT TO DATETIME 

df['Time_Orderd_dt'] = pd.to_datetime(df['Time_Orderd'], errors='coerce')
df['Time_Order_picked_dt'] = pd.to_datetime(df['Time_Order_picked'], errors='coerce')

# CREATE FEATURES

df['order_hour'] = df['Time_Orderd_dt'].dt.hour
df['pickup_hour'] = df['Time_Order_picked_dt'].dt.hour

# CREATE PREPARATION TIME (VERY IMPORTANT FEATURE)

df['order_prep_time'] = (df['Time_Order_picked_dt'] - df['Time_Orderd_dt']).dt.total_seconds() / 60

df[['Time_Orderd','Time_Order_picked','order_hour','pickup_hour','order_prep_time']].head(5)

,Time_Orderd,Time_Order_picked,order_hour,pickup_hour,order_prep_time
0,21:55,22:10,21.0,22.0,15.0
1,14:55,15:05,14.0,15.0,10.0
2,17:30,17:40,17.0,17.0,10.0
3,09:20,09:30,9.0,9.0,10.0
4,19:50,20:05,19.0,20.0,15.0


In [60]:
# CHECK MISSING

df[['order_hour','pickup_hour','order_prep_time']].isnull().sum()

order_hour         4345
pickup_hour        4274
order_prep_time    8150
dtype: int64

In [61]:
# FILL MISSING VALUES

df['order_hour'].fillna(df['order_hour'].median(), inplace=True)
df['pickup_hour'].fillna(df['pickup_hour'].median(), inplace=True)
df['order_prep_time'].fillna(df['order_prep_time'].median(), inplace=True)

# VERIFY

df[['order_hour','pickup_hour','order_prep_time']].isnull().sum()

order_hour         0
pickup_hour        0
order_prep_time    0
dtype: int64

In [62]:
# PEAK HOUR FEATURE

df['is_peak_hour'] = df['order_hour'].apply(lambda x: 1 if (11 <= x <= 14) or (18 <= x <= 22) else 0)

df[['order_hour','is_peak_hour']].head(10)

,order_hour,is_peak_hour
0,21.0,1
1,14.0,1
2,17.0,0
3,9.0,0
4,19.0,1
5,20.0,1
6,14.0,1
7,20.0,1
8,21.0,1
9,20.0,1


In [63]:
# MAP TRAFFIC LEVELS

traffic_map = { 'Low': 1,'Medium': 2,'High': 3,'Jam': 4}

df['traffic_level'] = df['Road_traffic_density'].map(traffic_map)

df[['Road_traffic_density','traffic_level']].head(10)

,Road_traffic_density,traffic_level
0,Jam,4
1,High,3
2,Medium,2
3,Low,1
4,Jam,4
5,Jam,4
6,High,3
7,Jam,4
8,Jam,4
9,Jam,4


In [64]:
# INTERACTION FEATURE

df['distance_traffic'] = df['distance_km'] * df['traffic_level']

df[['distance_km','traffic_level','distance_traffic']].head(10)

,distance_km,traffic_level,distance_traffic
0,10.280582,4,41.122328
1,6.242319,3,18.726956
2,13.787860,2,27.575720
3,2.930258,1,2.930258
4,19.396618,4,77.586473
5,13.763977,4,55.055909
6,6.218001,3,18.654004
7,16.849940,4,67.399760
8,4.540574,4,18.162298
9,12.256076,4,49.024306


In [65]:
# CHECK CATEGORICAL COLUMNS

cat_cols = df.select_dtypes(include='object').columns
cat_cols

Index(['Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weather_conditions',
       'Road_traffic_density', 'Type_of_order', 'Type_of_vehicle', 'Festival',
       'City'],
      dtype='object')

In [66]:
# ONE HOT ENCODING

df = pd.get_dummies(df, columns=['Weather_conditions', 'Type_of_order','Type_of_vehicle','City'], drop_first=True)

df.head()

,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Road_traffic_density,...,Weather_conditions_Stormy,Weather_conditions_Sunny,Weather_conditions_Windy,Type_of_order_Drinks,Type_of_order_Meal,Type_of_order_Snack,Type_of_vehicle_motorcycle,Type_of_vehicle_scooter,City_Semi-Urban,City_Urban
0,36.0,4.2,30.327968,78.046106,30.397968,78.116106,12-02-2022,21:55,22:10,Jam,...,0,0,0,0,0,1,1,0,0,0
1,21.0,4.7,10.003064,76.307589,10.043064,76.347589,13-02-2022,14:55,15:05,High,...,1,0,0,0,1,0,1,0,0,0
2,23.0,4.7,18.562450,73.916619,18.652450,74.006619,04-03-2022,17:30,17:40,Medium,...,0,0,0,1,0,0,0,1,0,0
3,34.0,4.3,30.899584,75.809346,30.919584,75.829346,13-02-2022,09:20,09:30,Low,...,0,0,0,0,0,0,1,0,0,0
4,24.0,4.7,26.463504,80.372929,26.593504,80.502929,14-02-2022,19:50,20:05,Jam,...,0,0,0,0,0,1,0,1,0,0


In [74]:
#LABEL ENCODING
df['Festival'] = df['Festival'].map({'Yes': 1, 'No': 0})

In [75]:
df.isnull().sum()

Delivery_person_Age                 0
Delivery_person_Ratings             0
Restaurant_latitude                 0
Restaurant_longitude                0
Delivery_location_latitude          0
Delivery_location_longitude         0
Order_Date                          0
Time_Orderd                       835
Time_Order_picked                   0
Road_traffic_density                0
Vehicle_condition                   0
multiple_deliveries                 0
Festival                            0
Time_taken (min)                    0
distance_km                         0
Time_Orderd_dt                   4345
Time_Order_picked_dt             4274
order_hour                          0
pickup_hour                         0
order_prep_time                     0
is_peak_hour                        0
traffic_level                       0
distance_traffic                    0
Weather_conditions_Fog              0
Weather_conditions_Sandstorms       0
Weather_conditions_Stormy           0
Weather_cond

In [76]:
# NULL VALUE TREATMENT WITH IMPUTATION WITH MEDIAN
# FILL IMPORTANT NUMERICAL NULLS

df['Delivery_person_Age'].fillna(df['Delivery_person_Age'].median(), inplace=True)
df['Delivery_person_Ratings'].fillna(df['Delivery_person_Ratings'].median(), inplace=True)



df[['Delivery_person_Age','Delivery_person_Ratings']].isnull().sum()

Delivery_person_Age        0
Delivery_person_Ratings    0
dtype: int64

In [77]:
# TRAIN - TEST SPLIT

from sklearn.model_selection import train_test_split

# TARGET
y = df['Time_taken (min)']

# FEATURES (final selection)
X = df.drop(columns=['Time_taken (min)','Order_Date','Time_Orderd','Time_Order_picked','Time_Orderd_dt','Time_Order_picked_dt',
                     'Road_traffic_density'])

# SPLIT
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape

((31171, 28), (7793, 28))

In [78]:
#LINEAR REGRESSION

from sklearn.linear_model import LinearRegression

# MODEL
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# PREDICT
y_pred = lr_model.predict(X_test)

In [79]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# MAPE (safe calculation)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)
print("MAPE:", mape)

MAE: 4.746437526820493
RMSE: 5.93851603988464
R2 Score: 0.5856233016864872
MAPE: 20.77608845244271


In [80]:
#RANDOM FOREST
from sklearn.ensemble import RandomForestRegressor

# MODEL
rf = RandomForestRegressor(n_estimators=100,max_depth=10,random_state=42,n_jobs=-1)

# TRAIN
rf.fit(X_train, y_train)

# PREDICT
y_pred_rf = rf.predict(X_test)

In [81]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)
mape_rf = np.mean(np.abs((y_test - y_pred_rf) / np.maximum(y_test, 1))) * 100

print("RF MAE:", mae_rf)
print("RF RMSE:", rmse_rf)
print("RF R2:", r2_rf)
print("RF MAPE:", mape_rf)

RF MAE: 3.347259746300911
RF RMSE: 4.213410270055831
RF R2: 0.7914035289048759
RF MAPE: 14.464235889034747


In [82]:
#IMPORTANT FEATURES
import pandas as pd

feature_importance = pd.DataFrame({ 'feature': X_train.columns, 'importance': rf.feature_importances_}).sort_values(by='importance', ascending=False)

feature_importance.head(10)

,feature,importance
1,Delivery_person_Ratings,0.236212
15,distance_traffic,0.230550
6,Vehicle_condition,0.117367
0,Delivery_person_Age,0.105052
19,Weather_conditions_Sunny,0.087185
14,traffic_level,0.051423
16,Weather_conditions_Fog,0.041365
9,distance_km,0.035693
7,multiple_deliveries,0.034613
8,Festival,0.013106


In [91]:
# EXTREME GRADIENT BOOSTING (XGB MODEL)

from xgboost import XGBRegressor

# MODEL
xgb = XGBRegressor(n_estimators=300,learning_rate=0.1,max_depth=6,subsample=0.8,colsample_bytree=0.8,random_state=42,n_jobs=-1,
    early_stopping_rounds=5)

# TRAIN with validation set
xgb.fit( X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

# PREDICT
y_pred_xgb = xgb.predict(X_test)

In [92]:
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)
mape_xgb = np.mean(np.abs((y_test - y_pred_xgb) / np.maximum(y_test, 1))) * 100

print("XGB MAE:", mae_xgb)
print("XGB RMSE:", rmse_xgb)
print("XGB R2:", r2_xgb)
print("XGB MAPE:", mape_xgb)

XGB MAE: 3.1850751973263156
XGB RMSE: 3.979524946611977
XGB R2: 0.8139190472216219
XGB MAPE: 13.736176842955903


In [93]:
# HYPER PARAMETER TUNING

from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

# PARAMETER GRID
param_dist = { 'n_estimators': [200, 300, 400, 500], 'learning_rate': [0.01, 0.05, 0.1], 'max_depth': [4, 5, 6, 7, 8],
    'subsample': [0.7, 0.8, 0.9],'colsample_bytree': [0.7, 0.8, 0.9],'gamma': [0, 0.1, 0.2],'min_child_weight': [1, 3, 5]}

# MODEL
xgb_base = XGBRegressor(random_state=42, n_jobs=-1)

# RANDOM SEARCH
random_search = RandomizedSearchCV(estimator=xgb_base,param_distributions=param_dist,
    n_iter=20, scoring='neg_mean_absolute_error', cv=3, verbose=1, random_state=42, n_jobs=-1)

# TRAIN
random_search.fit(X_train, y_train)

# BEST MODEL
best_xgb = random_search.best_estimator_

best_xgb

Fitting 3 folds for each of 20 candidates, totalling 60 fits


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0.1, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=3, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=-1, num_parallel_tree=None, ...)

In [94]:
y_pred_tuned = best_xgb.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_tuned)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
r2 = r2_score(y_test, y_pred_tuned)
mape = np.mean(np.abs((y_test - y_pred_tuned) / np.maximum(y_test, 1))) * 100

print("Tuned MAE:", mae)
print("Tuned RMSE:", rmse)
print("Tuned R2:", r2)
print("Tuned MAPE:", mape)

Tuned MAE: 3.0935525921087708
Tuned RMSE: 3.8672728725386563
Tuned R2: 0.8242687122745535
Tuned MAPE: 13.322274826000765


In [95]:
#CROSS VALIDATION

from sklearn.model_selection import cross_val_score

# CROSS VALIDATION (MAE)
cv_scores = cross_val_score(best_xgb,X,y,cv=10,scoring='neg_mean_absolute_error',n_jobs=-1)

# CONVERT TO POSITIVE
cv_mae = -cv_scores

print("CV MAE Scores:", cv_mae)
print("Mean CV MAE:", cv_mae.mean())
print("Std Dev:", cv_mae.std())

CV MAE Scores: [3.09138564 3.1459356  3.17625239 3.10062141 3.06205408 3.10877863
 3.1479606  3.07274206 3.09010719 3.12772879]
Mean CV MAE: 3.1123566378174976
Std Dev: 0.03449515906413007
